In [ ]:
conda install numpy pandas matplotlib seaborn scikit-learn lightgbm xgboost jupyter
pip install scipy polars pyarrow huggingface_hub

In [ ]:
pip install autogluon

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import polars as pl
from datetime import datetime, timedelta

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r1_score, log_loss, roc_curve

from scipy.signal import butter, lfilter, stft

from huggingface_hub import hf_hub_download
from datasets import load_dataset

from lightgbm import LGBMClassifier, LGBMRegressor
from xgboost import XGBClassifier, XGBRegressor

from autogluon.tabular import TabularDataset, TabularPredictor
from autogluon.timeseries import TimeSeriesDataFrame, TimeSeriesPredictor

# Data preprocess

### Pandas ver.

#### read and eda data

In [ ]:
df = pd.read_csv()

In [ ]:
display(df.head(5))
display(df.tail(5))
print(df.shape)
print(df.columns)
print(df.dtypes)

In [ ]:
print("df unique each columns:")
print(df.nunique())

print("df null each columns:")
print(df.isnull().sum())

In [ ]:
#Clean dataframe
df = df.drop(columns=['name', 'name'])

df['a'] = df['a'].fillna(0)
df['b'] = df['b'].fillna(df['a'].mean())
df['c'] = df['c'].fillna(df['c'].median())
df['d'] = df['d'].fillna('missing')
df['f'] = df['f'].fillna(method='ffill')

df = df[(df['a'] >= min_date) & (df['a'] <= max_date)]

df = df.dropna(inplace=True)

In [ ]:
#Data type
df['date'] = pd.to_datetime(df['date'], format="%Y-%m-%d")
df['date'] = pd.to_datetime(df[['year', 'month', 'day']])

min_date = df['date'].min() + timedelta(days=10)
max_date = df['date'].max() - timedelta(days=45)

df = df[(df['date'] >= pd.to_datetime("2025-02-01")) & (df['date'] <= pd.to_datetime("2025-04-03"))]

df['day'] = df['date'].dt.day
df['day_of_week'] = df['date'].dt.weekday  # 0 = Monday
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year

df['week'] = ((df['day'] - 1) // 7 + 1).clip(upper=4)

df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

In [ ]:
#Data manipulate
agg_df = df.groupby(['a', 'b', 'c']).agg({
    'target': 'sum',
    'd': 'first',
    'e': 'last',
    'f': 'min',
    'g': 'max',
    'h': 'median',
    'i': 'mean',
    'j': 'sum',
    'k': 'count',
    'l': pd.Series.nunique,
    'm': lambda x: list(x),
    'o': 'std',
    'p': lambda x: x[x > 0].min() if (x > 0).any() else np.nan
})

agg_df = agg_df.rename(columns={
    'target': 'target',
    'j': 'sum_j'
})

agg_df = agg_df.reset_index()

list_col = df['name'].unique()
df = df[df['name'].isin(list_col)]

In [ ]:
df = other_df.merge(other_df, how='cross')

df = df.merge(data, on=["a", "b", "c"], how="left")

df = pl.from_pandas(df)

### Polars ver.

#### read and eda data

In [ ]:
df = pl.read_csv("")

In [ ]:
display(df.head(5))
display(df.tail(5))
print(df.shape)
print(df.columns)
print(df.dtypes)

In [ ]:
# https://docs.pola.rs/user-guide/expressions/basic-operations/#counting-unique-values

print("df unique each columns:")
print(df.select(pl.all().n_unique()))

print("df null each columns:")
print(df.null_count())
print(df.select(pl.all().is_null().sum()))

In [ ]:
#Clean dataframe
df = df.drop(['name', 'name'])

df = df.with_columns([pl.col('a').fill_null(0).alias('name'),
                      pl.col('b').fill_null(pl.col('a').mean()).alias('name'),
                      pl.col('c').fill_null(pl.col('c').median()).alias('name'),
                      pl.col('d').fill_null('missing').alias('name'),
                      pl.col('e').fill_null(pl.lit('missing')).alias('name'),
                      pl.col('f').fill_null(strategy='ffill').alias('name')
                      ])

df = df.filter(
    (pl.col("a") >= pl.lit(min_date)) & (pl.col("a") <= pl.lit(max_date))
    )

In [ ]:
#Data type
df = df.with_columns(
      pl.col("date").str.strptime(pl.Date, "%Y-%m-%d").alias("date")
      )

df = df.with_columns(
      pl.date(pl.col("year"), pl.col("month"), pl.col("day")).alias("full_date")
      )

# df(pl.col("date").min()) -> Polars DataFrame (1 row, 1 col)
# .item() -> Python scalar value (e.g., datetime.date(2023, 1, 1))
min_date = df.select(pl.col("date").min()).item() + timedelta(days=10)
max_date = df.select(pl.col("date").max()).item() - timedelta(days=45)

df = df.filter(
    (pl.col("date").is_between(pl.lit("2025-02-01").cast(pl.Date), pl.lit("2025-04-03").cast(pl.Date)))
)

df = df.with_columns([
    pl.col("date").dt.day().alias("day"),
    pl.col("date").dt.weekday().alias("day_of_week"),
    pl.col("date").dt.month().alias("month"),
    pl.when(((pl.col("date").dt.day() - 1) // 7 + 1) > 4)
      .then(4)
      .otherwise((pl.col("date").dt.day() - 1) // 7 + 1)
      .alias("week"),
    pl.col("date").dt.year().alias("year")
])

df = df.with_columns([
    pl.col("day_of_week").map_elements(lambda x: np.sin(2 * np.pi * x / 7)).alias("day_of_week_sin"),
    pl.col("day_of_week").map_elements(lambda x: np.cos(2 * np.pi * x / 7)).alias("day_of_week_cos"),
    pl.col("month").map_elements(lambda x: np.sin(2 * np.pi * x / 12)).alias("month_sin"),
    pl.col("month").map_elements(lambda x: np.cos(2 * np.pi * x / 12)).alias("month_cos"),
])

In [ ]:
#Data manipulate
df = df.group_by(['a', 'b', 'c']).agg([
    pl.col('target').sum().alias('target'),
    pl.col('d').first(),
    pl.col('e').last(),
    pl.col('f').min(),
    pl.col('g').max(),
    pl.col('h').median(),
    pl.col('i').mean(),
    pl.col('j').sum().alias('sum_j'),
    pl.col('k').count(),
    pl.col('l').n_unique(),
    pl.col('m').list(),
    pl.col('n').implode(),
    pl.col('o').std(),
    pl.col('p').filter(pl.col('p') > 0).min()
])

list_col = df['name'].unique()
df = df.filter(pl.col('name').is_in(list_col))


In [ ]:
#Join df
#Cross join using 2 df
df = other_df.join(other_df, how="cross")

#Join on left
df = df.join(data, on=["a", "b", "c"], how="left")


In [ ]:
df = df.to_pandas()

### Time_series

In [ ]:
def to_timeseries(df_sales: pl.DataFrame) -> pl.DataFrame:
    # 1. Get min and max date
    min_date = df_sales.select(pl.col("OrderDate").min()).item()
    max_date = df_sales.select(pl.col("OrderDate").max()).item()

    # 2. Create calendar
    calendar_df = pl.DataFrame({
        "OrderDate": pl.date_range(min_date, max_date, interval="1d", eager=True)
    })

    # 3. Get frequent Customer-Product pairs
    pairs = (
        df_sales
        .group_by(["CustomerBKey", "ProductForPlan10", 'ProductForPlan1', 'ProductBKey', 'MODEL_4', 'ProductForPlan8'])
        .len()
        .filter(pl.col("len") > 5)
        .select(["CustomerBKey", "ProductForPlan10", 'ProductForPlan1', 'ProductBKey', 'MODEL_4', 'ProductForPlan8'])
    )

    # 4. Cross join pairs with dates to build time series base
    base_df = pairs.join(calendar_df, how="cross")

    sales_data = df_sales.select([
        'OrderDate', 'CustomerBKey', 'ProductForPlan10', 'OrderWeight'
    ]).with_columns(pl.col("OrderDate").cast(pl.Date))

    base_df = base_df.join(sales_data, on=["OrderDate", "CustomerBKey", "ProductForPlan10"], how="left")

    # 6. Fill missing OrderWeight with 0
    base_df = base_df.with_columns(
        pl.col("OrderWeight").fill_null(0.0).cast(pl.Float32)
    )

    return base_df

def to_timeseries(df: pl.DataFrame, date_column: str = "") -> pl.DataFrame:
    min_date = df.select(pl.col(f"{date_column}").min()).item()
    max_date = df.select(pl.col(f"{date_column}").max()).item()

    calendar_df = pl.DataFrame({
        date_column: pl.date_range(min_date, max_date, interval='1d', eager=True)
    })

    pairs = (
        df.group_by(['a', 'b', 'c']).len().filter(pl.col("len") > 0).select(['a', 'b', 'c'])
    )

    base_df = pairs.join(calendar_df, how='cross')
    df = df.select([date_column, 'a', 'b', 'c']).with_columns(pl.col(f"{date_column}").cast(pl.Date))

    base_df = base_df.join(df, on=[date_column, 'a', 'b', 'c'], how='left')

    base_df = base_df.with_columns(pl.col("target").fill_null(0.0).cast(pl.Float32))

    return base_df



In [ ]:
def cutout_zeros(df: pl.DataFrame, col: str = 'OrderWeight', fraction: float = 0.5, seed: int = 42) -> pl.DataFrame:
    """
    Randomly removes a fraction of rows where `col == 0`.

    Args:
        df (pl.DataFrame): The Polars DataFrame.
        col (str): Column name to evaluate (usually "sales").
        frac (float): Fraction (0.0 to 1.0) of zero-value rows to keep.
        seed (int): Random seed for reproducibility.

    Returns:
        pl.DataFrame: Filtered DataFrame with reduced zero-value rows.
    """
    np.random.seed(seed)

    # Count stats
    zero_rows = df.filter(pl.col(col) == 0)
    non_zero_rows = df.filter(pl.col(col) > 0)

    print(f"Original rows with {col} == 0: {len(zero_rows)}")
    print(f"Original rows with {col} >  0: {len(non_zero_rows)}")


    # Sample from zero rows
    n_samples = int(zero_rows.height * fraction)
    zero_sampled = zero_rows.sample(n=n_samples, seed=seed)

    # Combine back
    result = pl.concat([non_zero_rows, zero_sampled]).rechunk()
    print(f"After cutout: {result.filter(pl.col(col) == 0).height} rows with {col} == 0")
    print(f"Total rows after cutout: {result.height}")

    return result

In [ ]:
transaction1 = to_timeseries(transaction1)
transaction1 = cutout_zeros(transaction1, col='OrderWeight', fraction=1, seed=42)

# Signal

In [ ]:
#turn signal column to fft
#format output be like
'''
| id | column | frequency | amplitude |
| -- | ------ | --------- | --------- |
| 1  | value1 | 0.0       | 23.4      |
| 1  | value1 | 0.1       | 10.2      |
| 1  | value2 | 0.0       | 13.4      |
| 1  | value2 | 0.1       | 8.5       |
| 2  | value1 | 0.0       | 19.8      |
'''
def plot_frequency_domain_fft(df):
    sampling_rate = 2

    unique_id = df['id'].unique()
    column = []

    fft_results = []
    i = 0

    for id in unique_id:
      id_df = df[df['id'] == id].copy()
      i+=1
      for col in column:
          signal = id_df[col].values
          n = len(signal)

          fft_values = np.fft.fft(signal)
          frequencies = np.fft.fftfreq(n, d=1/sampling_rate)

          fft_df = pd.DataFrame({'id': id, 'column': col, 'frequency': frequencies, 'amplitude': np.abs(fft_values)})
          fft_results.append(fft_df)
          if i % 10 == 0:
              plt.figure(figsize=(10,6))
              plt.title(f"FFT of {col}")
              plt.plot(fft_df['frequency'], fft_df['amplitude'])
              plt.xlabel('Frequency')
              plt.ylabel('Amplitude')
              plt.show()

    #fft_df = pd.concat(fft_results, ignore_index=True)

In [ ]:
def plot_frequency_domain_stft(df):
    sampling_rate = 2

    unique_id = df['id'].unique()
    column = []

    stft_results = []
    i = 0

    for id in unique_id:
      id_df = df[df['id'] == id].copy()
      i+=1
      for col in column:
          signal = id_df[col].values
          n = len(signal)

          frequencies, times, stft_values = stft(signal, fs=sampling_rate)
          if i % 10 == 0:
                plt.figure(figsize=(10, 6))
                plt.pcolormesh(times, frequencies, np.abs(stft_values), shading='gouraud')
                plt.title(f"STFT of {col} (id={id})")
                plt.ylabel('Frequency [Hz]')
                plt.xlabel('Time [sec]')
                plt.colorbar(label='Amplitude')
                plt.tight_layout()
                plt.show()
          # Store STFT results as long-format DataFrame
      #     magnitude = np.abs(Zxx)
      #     for ti, time in enumerate(t):
      #          for fi, freq in enumerate(f):
      #             stft_results.append({
      #                   'id': id_,
      #                   'column': col,
      #                   'time': time,
      #                   'frequency': freq,
      #                   'amplitude': magnitude[fi, ti]
      #               })

      # stft_df = pd.DataFrame(stft_results)
      # return stft_df

In [ ]:
#Band pass filter function

#Signal = List of value you want to bandpass
def band_pass_filter(signal, sample_rate):
    n = len(signal)
    fft_signal = np.fft.fft(signal)
    frequencies = np.fft.fftfreq(n, d=1 / sample_rate)

    mask = frequencies > 0
    idx = np.argmax(np.abs(fft_signal[mask]))
    dominent_freq = frequencies[mask][idx]

    lowcut = max(0.01, dominent_freq - 1000)
    highcut = min(dominent_freq * 2, 16000)

    nyquist = 0.5 * sample_rate
    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(4, [low, high], btype='band')

    filtered_signal = lfilter(b, a, signal)

    return filtered_signal

# Model

#### Model combine in one function

In [ ]:
def autotrain(df, test_df, target, loop_col: str = 'a'):
    RANDOM_STATE = 42
    all_prediction = []
    target_loop = df[loop_col].unique()

    base_feature = [col for col in df.columns if col not in [loop_col, target]]

    categorical_feature = []

    df['have_target'] = (df[target] > 0).astype(int)

    for i in target_loop:
        print(f"Loop target: {i}")
        train_df = df[df[loop_col] == i].copy()
        predict_df = test_df[test_df[loop_col] == i].copy()

        x_item = train_df[base_feature]
        y_item = train_df[target]
        y_have_target = train_df['have_target']

        #Train classifier
        print(f"  Training classifier for {i}...")
        x_train_clf, x_val_clf, y_train_clf, y_val_clf = train_test_split(x_item, y_have_target, test_size=0.2, random_state=RANDOM_STATE)

        classifier = LGBMClassifier(
            objective='binary',
            metric='logloss',
            n_estimators=150,
            learning_rate=0.05,
            num_leaves=31,
            verbose=-1,
            random_state=RANDOM_STATE,
            n_jobs=-1
        )

        classifier.fit(
            x_train_clf,
            y_train_clf,
            eval_set=[(x_val_clf, y_val_clf)],
            eval_metric='logloss',
            categorical_feature=categorical_feature,
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        val_probs = classifier.predict_proba(x_val_clf)[:, 1]

        clf_logloss = log_loss(y_val_clf, val_probs)
        clf_auc = roc_auc_score(y_val_clf, val_probs)
        fpr, tpr, thresholds = roc_curve(y_val_clf, val_probs)
        optimal_threshold_roc = thresholds[np.argmax(tpr - fpr)]

        best_f1 = 0
        optimal_threshold_f1 = 0.5
        for t in np.linspace(0.01, 0.99, 99):
            preds = (val_probs >= t).astype(int)
            f1 = f1_score(y_val_clf, preds)
            if f1 > best_f1:
                best_f1 = f1
                optimal_threshold_f1 = t

        val_preds = (val_probs >= optimal_threshold_f1).astype(int)
        val_f1 = f1_score(y_val_clf, val_preds)
        val_preds = (val_probs >= optimal_threshold_roc).astype(int)
        val_roc = f1_score(y_val_clf, val_preds)

        print(f"  Classifier logloss: {clf_logloss}")
        print(f"  Classifier AUC: {clf_auc}")
        print(f"  Classifier f1: {val_f1}")
        print(f"  Classifier roc: {val_roc}")
        print(f"  Optimal threshold using roc: {optimal_threshold_roc}")
        print(f"  Optimal threshold using f1: {optimal_threshold_f1}")

        if val_f1 > val_roc:
            optimal_threshold = optimal_threshold_f1
        else:
            optimal_threshold = optimal_threshold_roc

        importance_df = pd.DataFrame({
            'feature': x_item.columns,
            'importance': classifier.feature_importances_
        })
        importance_df = importance_df.sort_values('importance', ascending=False)
        top5 = importance_df.head(5)
        print("  Top 5 Classifier features:")
        print(top5.to_string(index=False))

        train_df_prob = classifier.predict_proba(train_df)[:, 1]
        train_df_clf = (train_df_prob >= optimal_threshold).astype(int)
        train_df['have_target'] = train_df_clf
        train_df['predict_prob'] = train_df_prob

        predict_df_prob = classifier.predict_proba(predict_df)[:, 1]
        predict_df_clf = (predict_df_prob >= optimal_threshold).astype(int)
        predict_df['have_target'] = predict_df_clf
        predict_df['predict_prob'] = predict_df_prob

        #Train regressor
        print(f"  Training regressor for {i}...")

        regressor_features = base_feature + ['have_target', 'predict_prob']
        x_item = train_df[regressor_features]
        y_item = train_df[target]
        x_train_reg, x_val_reg, y_train_reg, y_val_reg = train_test_split(x_item, y_item, test_size=0.2, random_state=RANDOM_STATE)

        #objective='tweedie', 'regression'
        regressor = LGBMRegressor(
            objective='tweedie',
            tweedie_variance_power=1.5,
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            eval_metric='mae',
            verbose=-1
        )

        regressor.fit(
            x_train_reg,
            y_train_reg,
            eval_set=[(x_val_reg, y_val_reg)],
            categorical_feature=categorical_feature,
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )

        val_preds = regressor.predict(x_val_reg)
        val_preds = np.where(x_val_reg['have_target'] == 0, 0, val_preds)
        val_preds[val_preds < 0] = 0

        val_rmse = np.sqrt(mean_squared_error(y_val_reg, val_preds))
        val_mae = mean_absolute_error(y_val_reg, val_preds)

        print(f"  Regressor RMSE: {val_rmse}")
        print(f"  Regressor MAE: {val_mae}")

        important_df = pd.DataFrame({
            'feature': x_item.columns,
            'importance': regressor.feature_importances_
        })
        important_df = important_df.sort_values('importance', ascending=False)
        top5 = important_df.head(5)
        print("  Top 5 Regressor features:")
        print(top5.to_string(index=False))


        test_feature = predict_df[regressor_features]
        test_preds = regressor.predict(test_feature)
        test_preds = np.where(test_feature['have_target'] == 0, 0, test_preds)
        test_preds[test_preds < 0] = 0

        predict_df['predict'] = test_preds
        all_prediction.append(predict_df[['id', 'predict']])

    if all_prediction:
        if 'predict' in test_df.columns:
            test_df = test_df.drop(columns=['predict'])
        prediction_df = pd.concat(all_prediction, ignore_index=True)
        display(prediction_df.head(5))
        test_df = test_df.merge(prediction_df, on='id', how='left')
        test_df['predict'] = test_df['predict'].fillna(0)

    return test_df

#### Separate model

# Autogluon


#### Tabular

In [ ]:
label = 'target'

In [ ]:
predictor = TabularPredictor(label=label, eval_metric='mae').fit(train_data)

In [ ]:
predictor.refit_full()

In [ ]:
prediction = predictor.predict(test_data)

In [ ]:
features_importance = predictor.feature_importance(train_df)
features_importance

In [ ]:
test_df['predict'] = prediction

#### Time-series

In [ ]:
# for polars dataframe with timeseries data

def create_range_date_df(df, date_col):
    min_date = df[date_col].min()
    max_date = df[date_col].max()
    date_range = pl.datetime_range(start=min_date, end=max_date, time_unit='ns', eager=True)
    range_df = pl.DataFrame({date_col: date_range})
    return range_df

def create_time_series_df(df, date_col, id_col, num_random_data=0):
    range_df = create_range_date_df(df, date_col)
    if num_random_data > 0:
        unique_id_df = df[id_col].unique().to_frame(id_col).sample(num_random_data, random_state=42)
    else:
        unique_id_df = df[id_col].unique().to_frame(id_col)
    full_df = unique_id_df.join(range_df, how='cross')

    full_df = full_df.join(df, on=[id_col, date_col], how='left')
    full_df = full_df.fill_null(0)
    return full_df

In [ ]:
df = create_time_series_df(df, date_col='date', id_col='id')
df

In [ ]:
train_df = TimeSeriesDataFrame.from_data_frame(df, id_column='id', time_column='date')

In [ ]:
predictor = TimeSeriesPredictor(
    prediction_length=7,
    target='target',
    eval_metric='MAE',
    )

predictor.fit(train_df, presets='medium_quality', time_limit=600)

In [ ]:
test_df = create_time_series_df(test_df, date_col='date', id_col='id')
test_df

In [ ]:
test_df = TimeSeriesDataFrame.from_data_frame(test_df, id_column='id', time_column='date')

In [ ]:
predictions = predictor.predict(test_df)
predictions.head()

In [ ]:
predictor.plot(test_df, predictions, max_num_item_ids=4)

In [ ]:
predictor.leaderboard(test_df)